# Simulating Y2H screens

In `simulate_experiment.ipynb`, I focused primarily on FACS, as it is more straight forward to model via biophysics. Y2H is a bit more challenging as what happens inside the cell is a bit of a black box. Here, we'll do our best to derive something that we can at least extract some trends from.

In FACS, the transfer function was as follows: Stability + Binding -> Fluorescence -> Probability of sorting. 

For Y2H, it's Stability + Binding -> Transcription of HIS3 -> Histidin production -> Growth rate

As before, we have the probability of folding variant, v:

$$
P_{fold,v} = \frac{1}{1 + e^{\Delta G_{fold,v} / RT}}
$$

And the probability that a folded mutant is bound to its binding partner:

$$
P_{bind,v} = \frac{[L]^{n_{Hill}}}{[L]^{n_{Hill}} + (K_{d,v})^{n_{Hill}}}
$$

There's just one problem, which is that we don't know the concentration of L inside the cell. We'll come back to this later.

When bound together, the transcription factor is reconstituted. For now we will assume that HIS3 transcription isn't maxed-out before full saturation of binding. 

We will also assume that the HIS3 enzyme is the limiting factor in histidine synthesis, so that the transcriptional signal for HIS3 is directly proportional to the amount of histidine produced. 

With those assumptions, we can say that the limiting substrate concentration, [S], is proportional to concentration of active transcription factors (bound partners)

$$
[S] = \alpha \cdot [M]_{max} \cdot P_{fold} \cdot P_{bind}
$$

Where $[M]_{max}$ is the maximum concentration possible of the mutant if it were perfectly stable, and $\alpha$ is a proportionality constant.

To model cellular growth rate as a function of the limiting substrate, there exists the Monod equation:

$$
\mu = \mu_{max} \cdot \frac{[S]}{K_s + [S]}
$$

where $\mu$ is the growth rate, $\mu_{max}$ is the maximum growth rate, $[S]$ is the concentration of limiting substrate for growth and $K_s$ is the half-velocity constant, the value of $[S]$ when the growth rate is half of its maximum.

However, the growth rate can also be modified by the addition of the inhibitor, 3-AT. 3-AT is a competitive inhibitor. If you had infinite enzyme, you'd still have the maximum possible production rate, so the inhibitor is actually increasing the amount of enzyme required to achieve the same catalytic rate. We are also assuming in our system that increasing the amount of enzyme directly increases the amount of limiting substrate. So in effect, adding inhibitor directly increases the amount of enzyme needed to produce the same signal, increasing the amount of limiting substrate concentration potential needed. In other words, it increases $K_{s}$.

How does it change $K_{s}$? As a competitive inhibitor, it follows Michaelis-Menten kinetics:

$$
K_m^{app} = K_m \cdot (1 + \frac{[I]}{K_i})
$$

So to apply this to our model, we simply swap $K_m$ for $K_s$, which we are directly proportional by our assumptions, giving us the following for the growth rate:

$$
\mu = \mu_{max} \cdot \frac{[S]}{K_s \cdot (1 + \frac{[I]}{K_i} )+ [S]}
$$

The final abundance of the variant depends on its initial abundance in the library, its growth rate, and the time of selection.

$$
N_{final} = N_{initial} \cdot e^{\mu \cdot t}
$$

For sampling, we will specify a number of miniprep input cells ($N_{samp}$), so the variant abundance going into template extraction will be a multinomial distribution where the probability of selecting a variant is given by it's final frequency in the flask at harvest.

So, the probability of pinking any specific variant is:

$$
p_i = \frac{N_{final, v}}{N_{total}}
$$

and the number of cells that make it into the pellet for extraction is given by:

$$
N_{pellet} \sim \text{Multinomial}(n=N_{samp}, p={p_1,...p_k})
$$

Then from here, the steps in `simulate_experiment.ipynb` apply.